# Progressive distillation: DDIM 100 → 50 → 25

Здесь весь алгоритм виден в ячейках: targets, два шага teacher, один шаг student,
обычный PyTorch training loop, EMA, checkpoints и inference. Teacher — сохранённая
модель вращающихся лучей после 120 эпох. Student начинает с её весов.

Метод: [Salimans & Ho, Progressive Distillation for Fast Sampling of Diffusion Models,
ICLR 2022](https://arxiv.org/abs/2202.00512), Algorithm 2 и Appendix G.
Мы переносим схему на дискретный cosine scheduler Diffusers. Его последний
training timestep имеет очень маленький положительный signal-to-noise ratio;
в статье верхняя граница задаётся с нулевым SNR.

Откройте только этот `.ipynb` в Jupyter / Colab и выполните сверху вниз.
Первая ячейка устанавливает пакет из GitHub, следующая находит или клонирует
отдельные файлы весов `demo/pipeline`. Python 3.11+, интернет при установке.
По умолчанию выполняются **60 + 100 эпох** настоящего обучения на CPU.

In [ ]:
import subprocess
import sys

REPOSITORY_URL = "https://github.com/Vovanm88/ddpm_vibe_research.git"
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
    "diffusion-lab[interactive] @ git+" + REPOSITORY_URL,
])

In [ ]:
import tempfile
from pathlib import Path

PROJECT_ROOT = next(
    (root for root in (Path.cwd(), *Path.cwd().parents)
     if (root / "demo/pipeline/model_index.json").is_file()),
    None,
)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path(tempfile.mkdtemp(prefix="diffusion-lab-distill-")) / "repository"
    subprocess.check_call([
        "git", "clone", "--depth", "1", REPOSITORY_URL, str(PROJECT_ROOT),
    ])
PIPELINE_PATH = PROJECT_ROOT / "demo/pipeline"
print("Teacher files:", PIPELINE_PATH)

## Параметры исследования

Количество эпох, batch size, learning rate, EMA и dropout меняются здесь.
Сетка 100 → 50 → 25 совпадает с каждым вторым шагом предыдущей сетки.
Не заменяйте её на произвольные числа: ниже есть проверка вложенности и
соответствия настоящему `DDIMScheduler.step`.

`CONDITION_DROPOUT` обучает и Null-ветку: один и тот же keep mask используется
на обоих вызовах teacher и на student. Оценка проводится с guidance scale 1,
поэтому реальный NFE совпадает с числом шагов. CFG доступен в inference-функции,
но качество при других scale проверяйте отдельно.

In [ ]:
DEVICE = "cpu"  # "cuda", "mps"
NUM_THREADS = 4
SEED = 314
TRAIN_SEED = 123
REFERENCE_SEED = 2718
TRAIN_SAMPLES = 4096
REFERENCE_SAMPLES = 4096
BATCH_SIZE = 256
STAGE_STEPS = (100, 50, 25)
NUM_EPOCHS = (60, 100)
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.0
CONDITION_DROPOUT = 0.15
EMA_DECAY = 0.99
GRAD_CLIP = 1.0
EVAL_SAMPLES = 1024
EVAL_SEED = 1618
EVAL_GUIDANCE_SCALE = 1.0
TIMING_REPEATS = 3
CHECKPOINT_EVERY = 10
RESUME_IF_PRESENT = True
MAKE_GIFS = True
GIF_PARTICLES = 12
OUTPUT_DIR = Path("outputs/progressive_distillation")

In [ ]:
import copy
import hashlib
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from diffusers import DDIMScheduler
from torch.utils.data import DataLoader
from IPython.display import Image, display

from diffusion_lab import ResearchDiffusionPipeline
from diffusion_lab.environments import rotating_rays
from diffusion_lab.metrics import evaluate_samples
from diffusion_lab.visualization import plot_dataset, save_figure, save_trajectory_gif

torch.set_num_threads(NUM_THREADS)
torch.manual_seed(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
train_data = rotating_rays(n_samples=TRAIN_SAMPLES, n_classes=4, seed=TRAIN_SEED)
reference_data = rotating_rays(
    n_samples=REFERENCE_SAMPLES, n_classes=4, seed=REFERENCE_SEED,
)
source_pipeline = ResearchDiffusionPipeline.from_pretrained(
    PIPELINE_PATH, local_files_only=True, low_cpu_mem_usage=False,
).to(DEVICE)
base_scheduler = DDIMScheduler.from_config(source_pipeline.scheduler.config)
assert base_scheduler.config.prediction_type == "v_prediction"
assert base_scheduler.config.timestep_spacing == "trailing"
assert not base_scheduler.config.clip_sample and not base_scheduler.config.thresholding
assert base_scheduler.config.set_alpha_to_one
assert len(NUM_EPOCHS) == len(STAGE_STEPS) - 1
assert 0 <= CONDITION_DROPOUT < 1 and 0 <= EMA_DECAY < 1
initial_teacher = copy.deepcopy(source_pipeline.model).eval().requires_grad_(False)
print("Teacher:", type(initial_teacher).__name__, dict(initial_teacher.config))
print("Independent holdout:", REFERENCE_SEED, "training:", TRAIN_SEED)
plot_dataset(reference_data.samples, reference_data.conditions, title="Independent rotating rays")
plt.show()
plt.close()

## Индексы и два шага teacher

Обозначения: $\alpha_t=\sqrt{\bar\alpha_t}$, $\sigma_t=\sqrt{1-\bar\alpha_t}$.
Для v-prediction: $\hat x_0=\alpha_t z_t-\sigma_t\hat v$.
При `eta=0` один DDIM-переход имеет вид
$z_r=(\sigma_r/\sigma_t)z_t+
(\alpha_r-(\sigma_r/\sigma_t)\alpha_t)\hat x_0$.

Teacher делает `t → s → r`, student — `t → r`. `r=-1` — чистый финальный
сэмпл с $(\alpha_r,\sigma_r)=(1,0)$; это отдельная граница, не отрицательный
индекс массива. Сетка 100 шагов: `[999,989,…,9]`, 50: `[999,979,…,19]`.

In [ ]:
# Во всех функциях batch — первая ось; t/s/r — один int или [B] индексы.
alpha_bars = base_scheduler.alphas_cumprod.to(device=DEVICE, dtype=torch.float32)

def batch_times(timesteps, batch_size):
    values = torch.as_tensor(timesteps, device=DEVICE, dtype=torch.long)
    if values.ndim == 0:
        values = values.expand(batch_size)
    assert values.shape == (batch_size,)
    assert torch.all((values >= -1) & (values < len(alpha_bars)))
    return values

def coefficients(timesteps, sample):
    indices = batch_times(timesteps, len(sample))
    # clamp нужен только для безопасного gather: -1 заменяется alpha_bar=1.
    a_bar = alpha_bars[indices.clamp_min(0)]
    a_bar = torch.where(indices == -1, torch.ones_like(a_bar), a_bar)
    shape = (-1,) + (1,) * (sample.ndim - 1)
    return a_bar.sqrt().reshape(shape).to(sample), (1 - a_bar).sqrt().reshape(shape).to(sample)

def sampling_grid(steps):
    total = base_scheduler.config.num_train_timesteps
    assert isinstance(steps, int) and total % steps == 0, "Use divisors of training timesteps"
    scheduler = DDIMScheduler.from_config(base_scheduler.config)
    scheduler.set_timesteps(steps, device=DEVICE)
    return scheduler.timesteps.clone()

def stage_triplets(teacher_steps, student_steps):
    assert teacher_steps == 2 * student_steps
    teacher_grid, student_grid = sampling_grid(teacher_steps), sampling_grid(student_steps)
    assert torch.equal(student_grid, teacher_grid[::2])
    t = teacher_grid[::2]
    s = teacher_grid[1::2]
    r = torch.cat([teacher_grid[2::2], teacher_grid.new_tensor([-1])])
    assert torch.equal(r[:-1], student_grid[1:])
    assert torch.all(t > s) and torch.all(s > r)
    return t, s, r

def prediction_to_clean(noisy, prediction_v, timesteps):
    alpha, sigma = coefficients(timesteps, noisy)
    return alpha * noisy - sigma * prediction_v

def ddim_from_clean(noisy, clean_prediction, timestep, next_timestep):
    alpha_t, sigma_t = coefficients(timestep, noisy)
    alpha_r, sigma_r = coefficients(next_timestep, noisy)
    assert torch.all(sigma_t > 0)
    noise_ratio = sigma_r / sigma_t
    return noise_ratio * noisy + (alpha_r - noise_ratio * alpha_t) * clean_prediction

def ddim_step(model, noisy, timestep, next_timestep, conditions, condition_mask=None):
    t = batch_times(timestep, len(noisy))
    assert torch.all(t >= 0)
    predicted_v = model(noisy, t, conditions=conditions, condition_mask=condition_mask).sample
    clean_prediction = prediction_to_clean(noisy, predicted_v, t)
    return ddim_from_clean(noisy, clean_prediction, t, next_timestep)

for teacher_steps, student_steps in zip(STAGE_STEPS[:-1], STAGE_STEPS[1:]):
    t, s, r = stage_triplets(teacher_steps, student_steps)
    print(f"{teacher_steps} → {student_steps}: first {(int(t[0]), int(s[0]), int(r[0]))}, "
          f"last {(int(t[-1]), int(s[-1]), int(r[-1]))}")

## Distillation target — формула полностью в коде

После двух детерминированных шагов замороженного teacher получаем $z_r$.
Инвертируем один шаг student:

$$\widetilde x_0=\frac{z_r-(\sigma_r/\sigma_t)z_t}
{\alpha_r-(\sigma_r/\sigma_t)\alpha_t}, \qquad
\widetilde v=\frac{\alpha_t z_t-\widetilde x_0}{\sigma_t}.$$

`MSE(student_v, target_v)` соответствует SNR+1-взвешенному MSE в x-space.
Teacher и target не участвуют в backprop. Ни clean training sample, ни noise
не используются как distillation target: они нужны, чтобы получить начальный $z_t$.

In [ ]:
@torch.no_grad()
def distillation_target(teacher, noisy, t, s, r, conditions, condition_mask=None):
    teacher.eval()
    intermediate = ddim_step(teacher, noisy, t, s, conditions, condition_mask)
    teacher_next = ddim_step(teacher, intermediate, s, r, conditions, condition_mask)
    alpha_t, sigma_t = coefficients(t, noisy)
    alpha_r, sigma_r = coefficients(r, noisy)
    noise_ratio = sigma_r / sigma_t
    denominator = alpha_r - noise_ratio * alpha_t
    assert torch.all(denominator > 0), "The nested DDIM interval must be nondegenerate"
    target_clean = (teacher_next - noise_ratio * noisy) / denominator
    target_v = (alpha_t * noisy - target_clean) / sigma_t
    return target_v.detach(), target_clean.detach(), teacher_next.detach()

# Проверяем индексную математику по официальному scheduler и target identity.
test_generator = torch.Generator().manual_seed(SEED + 1)
test_noisy = torch.randn(12, 2, generator=test_generator).to(DEVICE)
test_conditions = (torch.arange(12) % 4).to(DEVICE)
with torch.no_grad():
    for steps in STAGE_STEPS:
        official = DDIMScheduler.from_config(base_scheduler.config)
        official.set_timesteps(steps, device=DEVICE)
        grid = official.timesteps
        for index in (0, len(grid) // 2, len(grid) - 1):
            t = grid[index]
            r = grid[index + 1] if index + 1 < len(grid) else -1
            v = initial_teacher(test_noisy, t, conditions=test_conditions).sample
            ours = ddim_from_clean(test_noisy, prediction_to_clean(test_noisy, v, t), t, r)
            reference = official.step(v, t, test_noisy, eta=0).prev_sample
            torch.testing.assert_close(ours, reference, rtol=2e-5, atol=2e-6)
    for teacher_steps, student_steps in zip(STAGE_STEPS[:-1], STAGE_STEPS[1:]):
        t, s, r = stage_triplets(teacher_steps, student_steps)
        indices = torch.tensor([0, len(t) // 2, len(t) - 1], device=DEVICE).repeat(4)
        target_v, target_x0, teacher_next = distillation_target(
            initial_teacher, test_noisy, t[indices], s[indices], r[indices], test_conditions,
        )
        implied_next = ddim_from_clean(test_noisy, target_x0, t[indices], r[indices])
        torch.testing.assert_close(implied_next, teacher_next, rtol=2e-5, atol=2e-6)
        assert not target_v.requires_grad
print("Official DDIM indexing and two-to-one target identity: passed")

## Inference с явным числом вызовов модели

Все варианты используют один и тот же initial noise и class labels. Никаких
скрытых scheduler шагов или повторного зашумления. History содержит initial noise
и реальные выходы каждого шага. CFG не «запекается» в targets: по умолчанию
teacher/student вычисляют guidance 1. При другом scale функция считает
дополнительные unconditional-вызовы в NFE.

In [ ]:
@torch.no_grad()
def inference(model, steps, initial_noise, conditions=None, guidance_scale=1.0, record_history=False):
    assert guidance_scale >= 0 and np.isfinite(guidance_scale)
    model.eval()
    grid = sampling_grid(steps)
    sample = initial_noise.detach().clone().to(DEVICE)
    conditions = None if conditions is None else conditions.to(DEVICE)
    history = [sample.cpu().clone()] if record_history else None
    nfe = 0
    for index, timestep in enumerate(grid):
        next_time = grid[index + 1] if index + 1 < len(grid) else -1
        predicted_v = model(sample, timestep, conditions=conditions).sample
        nfe += 1
        if conditions is not None and guidance_scale != 1:
            unconditioned_v = model(sample, timestep, conditions=None).sample
            nfe += 1
            predicted_v = unconditioned_v + guidance_scale * (predicted_v - unconditioned_v)
        clean = prediction_to_clean(sample, predicted_v, timestep)
        sample = ddim_from_clean(sample, clean, timestep, next_time)
        if record_history:
            history.append(sample.cpu().clone())
    return {"samples": sample.cpu(), "history": torch.stack(history) if history is not None else None,
            "timesteps": grid.cpu(), "nfe": nfe}

eval_generator = torch.Generator().manual_seed(EVAL_SEED)
eval_noise = torch.randn(EVAL_SAMPLES, 2, generator=eval_generator)
eval_conditions = torch.arange(EVAL_SAMPLES) % 4
teacher_result = inference(initial_teacher, STAGE_STEPS[0], eval_noise, eval_conditions,
                           EVAL_GUIDANCE_SCALE, record_history=True)
initial_baselines = {}
for steps in STAGE_STEPS[1:]:
    initial_baselines[steps] = inference(initial_teacher, steps, eval_noise, eval_conditions,
                                       EVAL_GUIDANCE_SCALE)
print("Teacher NFE:", teacher_result["nfe"], "original teacher, fewer steps:",
      {steps: result["nfe"] for steps, result in initial_baselines.items()})

## Обычный PyTorch training loop: 100 → 50, затем 50 → 25

Для каждого batch выбирается один interval из **student**-сетки, добавляется
новый Gaussian noise, teacher делает два шага с тем же keep mask. Затем видны
`zero_grad → forward → loss → backward → clip → optimizer.step`.

EMA также обновляется явно. После stage замораживаем EMA student и используем
его как teacher следующего stage. Checkpoint содержит live/EMA веса, AdamW,
историю и RNG всех четырёх генераторов: DataLoader, noise, timestep и mask.
При повторном запуске ячеек обучение продолжается с последнего checkpoint.
Если меняете loss, архитектуру или рецепт, задайте новый `OUTPUT_DIR`;
resume проверяет параметры и hash teacher.

In [ ]:
training_history = []
stage_models = {}
teacher = copy.deepcopy(initial_teacher).eval().requires_grad_(False)
distillation_started = time.perf_counter()

for stage_index, (teacher_steps, student_steps, epochs) in enumerate(
    zip(STAGE_STEPS[:-1], STAGE_STEPS[1:], NUM_EPOCHS)
):
    stage_dir = OUTPUT_DIR / f"stage_{student_steps}"
    stage_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = stage_dir / "checkpoint.pt"
    t_grid, s_grid, r_grid = stage_triplets(teacher_steps, student_steps)
    teacher.eval().requires_grad_(False)
    student = copy.deepcopy(teacher).train().requires_grad_(True)
    ema_student = copy.deepcopy(student).eval().requires_grad_(False)
    optimizer = torch.optim.AdamW(student.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    loader_generator = torch.Generator().manual_seed(SEED + stage_index * 10 + 2)
    noise_generator = torch.Generator().manual_seed(SEED + stage_index * 10 + 3)
    timestep_generator = torch.Generator().manual_seed(SEED + stage_index * 10 + 4)
    mask_generator = torch.Generator().manual_seed(SEED + stage_index * 10 + 5)
    loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, drop_last=False,
                        generator=loader_generator, num_workers=0)
    stage_history = []
    first_epoch = 0
    teacher_digest = hashlib.sha256()
    for name, tensor in sorted(teacher.state_dict().items()):
        teacher_digest.update(name.encode())
        teacher_digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    run_contract = {
        "teacher_steps": teacher_steps, "student_steps": student_steps,
        "t": t_grid.cpu().tolist(), "s": s_grid.cpu().tolist(), "r": r_grid.cpu().tolist(),
        "seed": SEED, "train_seed": TRAIN_SEED, "train_samples": TRAIN_SAMPLES,
        "batch_size": BATCH_SIZE, "condition_dropout": CONDITION_DROPOUT,
        "ema_decay": EMA_DECAY, "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY, "teacher_guidance_scale": 1.0,
        "teacher_state_sha256": teacher_digest.hexdigest(),
    }
    if RESUME_IF_PRESENT and checkpoint_path.exists():
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
        assert checkpoint["run_contract"] == run_contract, "Checkpoint parameters changed"
        first_epoch = int(checkpoint["epoch"])
        if first_epoch > epochs:
            raise ValueError(
                f"Checkpoint already contains {first_epoch} epochs, but this stage requests {epochs}. "
                "Use a new OUTPUT_DIR or request at least the completed epoch count."
            )
        student.load_state_dict(checkpoint["student"])
        ema_student.load_state_dict(checkpoint["ema_student"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        torch.set_rng_state(checkpoint["torch_rng"].cpu())
        for generator, key in ((loader_generator, "loader_rng"), (noise_generator, "noise_rng"),
                               (timestep_generator, "timestep_rng"), (mask_generator, "mask_rng")):
            generator.set_state(checkpoint[key].cpu())
        if DEVICE.startswith("cuda") and checkpoint.get("cuda_rng") is not None:
            torch.cuda.set_rng_state_all(checkpoint["cuda_rng"])
        stage_history = checkpoint["history"]
        print(f"Resume {teacher_steps} → {student_steps} at epoch {first_epoch}")
    stage_started = time.perf_counter()
    previous_stage_seconds = stage_history[-1]["elapsed_seconds"] if stage_history else 0.0

    for epoch in range(first_epoch, epochs):
        student.train()
        epoch_loss = 0.0
        example_count = 0
        for batch in loader:
            clean = batch["samples"].to(DEVICE)
            conditions = batch["conditions"].to(DEVICE)
            indices = torch.randint(student_steps, (len(clean),), generator=timestep_generator).to(DEVICE)
            t, s, r = t_grid[indices], s_grid[indices], r_grid[indices]
            noise = torch.randn(clean.shape, generator=noise_generator, dtype=clean.dtype).to(DEVICE)
            alpha_t, sigma_t = coefficients(t, clean)
            noisy = alpha_t * clean + sigma_t * noise
            keep_condition = (torch.rand(len(clean), generator=mask_generator) >= CONDITION_DROPOUT).to(DEVICE)
            # Both teacher forwards and the student use this SAME mask.
            target_v, target_x0, target_next = distillation_target(
                teacher, noisy, t, s, r, conditions, keep_condition,
            )
            optimizer.zero_grad(set_to_none=True)
            predicted_v = student(noisy, t, conditions=conditions, condition_mask=keep_condition).sample
            loss = F.mse_loss(predicted_v, target_v)
            assert torch.isfinite(loss)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), GRAD_CLIP)
            optimizer.step()
            with torch.no_grad():
                for average, current in zip(ema_student.parameters(), student.parameters()):
                    average.lerp_(current.detach(), 1 - EMA_DECAY)
                for average, current in zip(ema_student.buffers(), student.buffers()):
                    average.copy_(current)
            epoch_loss += float(loss.detach()) * len(clean)
            example_count += len(clean)
        row = {"stage": f"{teacher_steps}->{student_steps}", "epoch": epoch + 1,
               "loss": epoch_loss / example_count,
               "elapsed_seconds": previous_stage_seconds + time.perf_counter() - stage_started}
        stage_history.append(row)
        if epoch == first_epoch or (epoch + 1) % 10 == 0 or epoch + 1 == epochs:
            print(f"{teacher_steps} → {student_steps}, epoch {epoch+1:3d}/{epochs}: "
                  f"MSE(v)={row['loss']:.7f}", flush=True)
        if (epoch + 1) % CHECKPOINT_EVERY == 0 or epoch + 1 == epochs:
            checkpoint = {
                "epoch": epoch + 1, "student": student.state_dict(),
                "ema_student": ema_student.state_dict(), "optimizer": optimizer.state_dict(),
                "run_contract": run_contract, "history": stage_history,
                "torch_rng": torch.get_rng_state(),
                "cuda_rng": torch.cuda.get_rng_state_all() if DEVICE.startswith("cuda") else None,
                "loader_rng": loader_generator.get_state(), "noise_rng": noise_generator.get_state(),
                "timestep_rng": timestep_generator.get_state(), "mask_rng": mask_generator.get_state(),
            }
            torch.save(checkpoint, checkpoint_path)
    # This trained EMA student becomes the frozen next-stage teacher.
    distilled = copy.deepcopy(ema_student).eval().requires_grad_(False)
    stage_models[student_steps] = distilled
    teacher = copy.deepcopy(distilled).eval().requires_grad_(False)
    training_history.extend(stage_history)
    ResearchDiffusionPipeline(distilled, DDIMScheduler.from_config(base_scheduler.config)).save_pretrained(
        stage_dir / "pipeline", safe_serialization=True,
    )
    distillation_config = {
        **run_contract, "completed_epochs": epochs, "exported_weights": "ema",
        "expected_inference_steps": student_steps, "eta": 0.0,
        "paper": "https://arxiv.org/abs/2202.00512",
    }
    (stage_dir / "distillation_config.json").write_text(json.dumps(distillation_config, indent=2))
    (stage_dir / "pipeline/distillation_config.json").write_text(json.dumps(distillation_config, indent=2))
    (stage_dir / "history.json").write_text(json.dumps(stage_history, indent=2))

training_seconds = time.perf_counter() - distillation_started
total_training_seconds = sum(
    json.loads((OUTPUT_DIR / f"stage_{steps}/history.json").read_text())[-1]["elapsed_seconds"]
    for steps in STAGE_STEPS[1:]
)
assert all(parameter.grad is None for parameter in initial_teacher.parameters())
assert all(not parameter.requires_grad for parameter in initial_teacher.parameters())
print(f"Distillation complete: {training_seconds:.1f}s; exported stages {list(stage_models)}")

## Сравнение на одинаковом noise и независимом holdout

Сравниваем исходный teacher на 100, 50 и 25 шагах и обученных students на 50/25.
SWD измеряет отличие от данных; paired RMSE — отличие от teacher-100 при том же
initial noise и label. Меньше — лучше. Низкий training loss сам по себе не
гарантирует качество многократного inference. Timing — медиана нескольких
полных запусков; NFE — фактические вызовы сети.

In [ ]:
results = {f"Teacher {STAGE_STEPS[0]}": teacher_result}
models_and_budgets = {f"Teacher {STAGE_STEPS[0]}": (initial_teacher, STAGE_STEPS[0])}
for steps in STAGE_STEPS[1:]:
    results[f"Original teacher {steps}"] = initial_baselines[steps]
    models_and_budgets[f"Original teacher {steps}"] = (initial_teacher, steps)
    results[f"Student {steps}"] = inference(
        stage_models[steps], steps, eval_noise, eval_conditions, EVAL_GUIDANCE_SCALE,
        record_history=True,
    )
    models_and_budgets[f"Student {steps}"] = (stage_models[steps], steps)

def synchronize():
    if DEVICE.startswith("cuda"):
        torch.cuda.synchronize()
    elif DEVICE == "mps":
        torch.mps.synchronize()

metric_rows = []
for name, result in results.items():
    values = evaluate_samples(reference_data, result["samples"], eval_conditions, seed=EVAL_SEED)
    class_swd = [value for key, value in values.items() if key.startswith("class_")]
    row = {"name": name, "nfe": result["nfe"],
           "sliced_wasserstein": values["sliced_wasserstein"],
           "mean_class_swd": float(np.mean(class_swd)),
           "paired_teacher_rmse": float((result["samples"] - teacher_result["samples"]).square().sum(-1).mean().sqrt())}
    model, steps = models_and_budgets[name]
    inference(model, steps, eval_noise, eval_conditions, EVAL_GUIDANCE_SCALE)  # warm-up
    elapsed = []
    for _ in range(TIMING_REPEATS):
        synchronize()
        start = time.perf_counter()
        inference(model, steps, eval_noise, eval_conditions, EVAL_GUIDANCE_SCALE)
        synchronize()
        elapsed.append(time.perf_counter() - start)
    row["median_sampling_seconds"] = float(np.median(elapsed))
    row["timing_runs_seconds"] = elapsed
    metric_rows.append(row)
    print(f"{name:20s} NFE={row['nfe']:3d} SWD={row['sliced_wasserstein']:.5f} "
          f"classSWD={row['mean_class_swd']:.5f} pairedRMSE={row['paired_teacher_rmse']:.5f} "
          f"time={row['median_sampling_seconds']:.3f}s")
    np.savez_compressed(OUTPUT_DIR / (name.lower().replace(" ", "_") + ".npz"),
                        samples=result["samples"].numpy(), conditions=eval_conditions.numpy(),
                        noise=eval_noise.numpy(), timesteps=result["timesteps"].numpy(),
                        **({"history": result["history"].numpy()} if result["history"] is not None else {}))

teacher_hash = hashlib.sha256((PIPELINE_PATH / "model/diffusion_pytorch_model.safetensors").read_bytes()).hexdigest()
report = {
    "paper": "https://arxiv.org/abs/2202.00512", "teacher_source": str(PIPELINE_PATH),
    "teacher_weights_sha256": teacher_hash, "teacher_original_training_epochs": 120,
    "stages": list(STAGE_STEPS), "epochs_per_stage": list(NUM_EPOCHS),
    "training_seconds_this_execution": training_seconds,
    "training_seconds_total": total_training_seconds,
    "seed": SEED, "train_seed": TRAIN_SEED, "reference_seed": REFERENCE_SEED,
    "eval_seed": EVAL_SEED, "eval_samples": EVAL_SAMPLES,
    "eval_guidance_scale": EVAL_GUIDANCE_SCALE, "condition_dropout": CONDITION_DROPOUT,
    "ema_decay": EMA_DECAY, "device": DEVICE,
    "torch_version": torch.__version__, "metrics": metric_rows,
    "interpretation": "Student learned two-to-one deterministic DDIM targets on its own timestep grid; quality is empirical.",
}
(OUTPUT_DIR / "metrics.json").write_text(json.dumps(report, indent=2))
(OUTPUT_DIR / "history.json").write_text(json.dumps(training_history, indent=2))

In [ ]:
selected_names = [f"Teacher {STAGE_STEPS[0]}"] + [f"Student {steps}" for steps in STAGE_STEPS[1:]]
fig, axes = plt.subplots(1, len(selected_names) + 1, figsize=(14, 4), layout="constrained")
plot_dataset(reference_data.samples, reference_data.conditions, title="Independent target", ax=axes[0], size=5)
for ax, name in zip(axes[1:], selected_names):
    row = next(row for row in metric_rows if row["name"] == name)
    plot_dataset(results[name]["samples"], eval_conditions, ax=ax, size=5,
                 title=f"{name} · NFE {row['nfe']}\nSWD {row['sliced_wasserstein']:.4f}")
for ax in axes:
    ax.set(xlim=(-1.2, 1.2), ylim=(-1.2, 1.2), aspect="equal")
    legend = ax.get_legend()
    if legend is not None:
        legend.remove()
fig.suptitle("Progressive distillation · matched noise and conditions")
save_figure(fig, OUTPUT_DIR / "comparison")
plt.show()
plt.close(fig)

fig, axes = plt.subplots(1, len(NUM_EPOCHS), figsize=(11, 3.6), layout="constrained", squeeze=False)
for ax, steps in zip(axes[0], STAGE_STEPS[1:]):
    stage = json.loads((OUTPUT_DIR / f"stage_{steps}/history.json").read_text())
    ax.plot([row["epoch"] for row in stage], [row["loss"] for row in stage], color="#2563eb")
    ax.set(title=f"Student {steps}", xlabel="Epoch", ylabel="Distillation MSE(v)", yscale="log")
    ax.grid(alpha=.15)
save_figure(fig, OUTPUT_DIR / "learning")
plt.show()
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
names = [row["name"] for row in metric_rows]
positions = np.arange(len(names))
axes[0].bar(positions - .18, [row["sliced_wasserstein"] for row in metric_rows], .36, label="Global SWD")
axes[0].bar(positions + .18, [row["mean_class_swd"] for row in metric_rows], .36, label="Mean condition SWD")
axes[0].set(ylabel="Distance · lower is better", title="Quality on heldout rays")
axes[0].legend(frameon=False)
axes[1].bar(positions, [row["median_sampling_seconds"] for row in metric_rows], color="#059669")
axes[1].set(ylabel="Seconds / full sampling run", title=f"Median of {TIMING_REPEATS} runs · {EVAL_SAMPLES} samples")
for ax in axes:
    ax.set_xticks(positions, names, rotation=25, ha="right")
    ax.grid(axis="y", alpha=.15)
save_figure(fig, OUTPUT_DIR / "quality_and_timing")
plt.show()
plt.close(fig)

## Радужные реальные траектории и повторное использование

GIF строится из сохранённого history каждого sampler. Цвет `turbo_r` показывает
продвижение от шума к сэмплу; Start/Finish отмечены. Число кадров ограничено
для размера файла, выбираются реальные сохранённые состояния.

Student предназначен для своего бюджета 50 или 25 и `eta=0`. Чтобы идти дальше,
нужны вложенные интервалы и следующая стадия обучения. Для следующего исследования
меняйте тело `distillation_target`, loss, sampling t, model или optimizer;
фиксируйте одинаковые latents, class labels и reference seed для сравнения.

In [ ]:
if MAKE_GIFS:
    for name in selected_names:
        paths = results[name]["history"][:, :GIF_PARTICLES].permute(1, 0, 2)
        gif_path = save_trajectory_gif(
            paths, OUTPUT_DIR / (name.lower().replace(" ", "_") + "_denoising.gif"),
            cmap="turbo_r", fps=10, max_frames=45, dpi=75,
            title=f"{name} · actual DDIM states", start_label="Noise x_T", end_label="Sample x_0",
        )
        print(name, gif_path)
        display(Image(filename=str(gif_path)))

# Загружаем stage отдельно, без обучения и без training wrapper.
student_steps = STAGE_STEPS[-1]
reloaded = ResearchDiffusionPipeline.from_pretrained(
    OUTPUT_DIR / f"stage_{student_steps}/pipeline", local_files_only=True,
    low_cpu_mem_usage=False,
).to(DEVICE)
reproduced = inference(reloaded.model, student_steps, eval_noise, eval_conditions, EVAL_GUIDANCE_SCALE)
torch.testing.assert_close(reproduced["samples"], results[f"Student {student_steps}"]["samples"])
# Generic Diffusers pipeline also needs this EXPLICIT trained inference budget.
reloaded.set_progress_bar_config(disable=True)
pipeline_samples = reloaded(
    batch_size=EVAL_SAMPLES, conditions=eval_conditions.to(DEVICE),
    num_inference_steps=student_steps, guidance_scale=EVAL_GUIDANCE_SCALE,
    generator=torch.Generator().manual_seed(EVAL_SEED), eta=0.0,
).samples.cpu()
torch.testing.assert_close(pipeline_samples, reproduced["samples"], rtol=2e-5, atol=2e-5)
print("Reload reproduces saved student samples; all outputs:", OUTPUT_DIR)